# Librerias

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de la bbdd y reducción

## 1. Cargar csv

In [13]:
#cargar datos
df = pd.read_parquet("../../data/df_limpiado_260427.parquet")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,contactado_campania_previa
0,59,administration,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,NaN,0,sin_campania_previa,yes,0
1,56,administration,married,secondary,no,45,no,no,unknown,5,may,1467,1,NaN,0,sin_campania_previa,yes,0
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,NaN,0,sin_campania_previa,yes,0
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,NaN,0,sin_campania_previa,yes,0
4,54,administration,married,tertiary,no,184,no,no,unknown,5,may,673,2,NaN,0,sin_campania_previa,yes,0


In [14]:
#Nos quedamos con las variables que usaremos para perfil de cliente
cols = ["age", "job", "marital", "education", "balance", "default", "housing", "loan", "deposit"]

df_perfil = df[cols].copy()
df_perfil.info()

<class 'pandas.DataFrame'>
RangeIndex: 10987 entries, 0 to 10986
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        10987 non-null  int64 
 1   job        10987 non-null  string
 2   marital    10987 non-null  string
 3   education  10987 non-null  string
 4   balance    10987 non-null  int64 
 5   default    10987 non-null  string
 6   housing    10987 non-null  string
 7   loan       10987 non-null  string
 8   deposit    10987 non-null  string
dtypes: int64(2), string(7)
memory usage: 1.1 MB


## 2. Categorizar edad

In [15]:
df_perfil["age_group"] = pd.cut(df_perfil["age"], bins=[0, 25, 35, 45, 55, 65, 120], labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"])

#Eliminamos la columna numérica original
df_perfil = df_perfil.drop(columns=["age"])

## 3. Categorizar multiproducto

In [16]:
#Crear variables binarias 0/1 para cada producto
df_perfil = df_perfil.copy()

df_perfil["deposit_bin"] = (df_perfil["deposit"] == "yes").astype(int)
df_perfil["loan_bin"] = (df_perfil["loan"] == "yes").astype(int)
df_perfil["housing_bin"] = (df_perfil["housing"] == "yes").astype(int)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1


In [17]:
#Crear variable combinación, según orden de 0 y 1 se puede saber que productos
def multiproducto(row):
    return f"{row['deposit_bin']}{row['loan_bin']}{row['housing_bin']}"

df_perfil["multiproducto"] = df_perfil.apply(multiproducto, axis=1)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101


In [18]:
#Crear columna catagorica de multiproducto:
etiquetas = {
    "000": "Ningún producto",
    "100": "Solo depósitos",
    "010": "Solo préstamos",
    "001": "Solo hipotecas",
    "110": "Depósitos y préstamos",
    "101": "Depósitos y hipotecas",
    "011": "Préstamos y hipotecas",
    "111": "Todos los productos"
}

df_perfil['multiproducto_cat'] = df_perfil['multiproducto'].astype(str).map(etiquetas)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto,multiproducto_cat
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101,Depósitos y hipotecas
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100,Solo depósitos
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101,Depósitos y hipotecas


In [19]:
#Crear columna conteo de productos:
df_perfil["cantidad_prod"] = df_perfil["deposit_bin"] + df_perfil["loan_bin"] + df_perfil["housing_bin"]

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto,multiproducto_cat,cantidad_prod
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101,Depósitos y hipotecas,2
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100,Solo depósitos,1
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101,Depósitos y hipotecas,2


In [ ]:
#Nos quedamos con variables para estudiar productos
cols_prod = ["age_group", "job", "marital", "education", "multiproducto", "multiproducto_cat", "cantidad_prod"]

df_product = df_perfil[cols_prod].copy()

df_product.head(3)

,age_group,job,marital,education,multiproducto,multiproducto_cat,cantidad_prod
0,56-65,administration,married,secondary,101,Depósitos y hipotecas,2
1,56-65,administration,married,secondary,100,Solo depósitos,1
2,36-45,technician,married,secondary,101,Depósitos y hipotecas,2


## 7. Conclusiones